In [29]:
import os
from bs4 import BeautifulSoup
import time
import urllib.request
from datetime import datetime, timedelta


In [30]:
CURRENT_TIME = datetime.now()
SEASONS = [x for x in range(2016,CURRENT_TIME.year+1)]
DATA_DIR = "data"
STANDINGS_DIR = os.path.join(DATA_DIR,"standings")
SCORES_DIR = os.path.join(DATA_DIR,"scores")
ACTIVE_SEASON = CURRENT_TIME.year if CURRENT_TIME.month < 7 else CURRENT_TIME.year+1

In [31]:
def get_html(url,selector, sleep_time = 5, retries=3):
    html = None
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    for i in range(1, retries+1):
        time.sleep(sleep_time*i)
        try:
            req = urllib.request.Request(url,headers=headers)
            with urllib.request.urlopen(req) as response:
                html = response.read().decode('utf-8')

                soup = BeautifulSoup(html, "html.parser")
                element = soup.select_one(selector)
                if element:
                    # Return the inner HTML of the selected element
                    # print(f"Successfully scraped {url} with selector {selector}")
                    return str(element)
                else:
                    print(f"Selector '{selector}' not found on {url}")
                    return None
        
        except Exception as e:
            print(f"Attempt {i} failed for {url}: {e}")
            if i < retries:
                print(f"Retrying in {sleep_time*i} seconds...")
            else:
                print("Max retries reached")
                return None

In [32]:
def scrape_season(season):
    url = f"https://www.basketball-reference.com/leagues/NBA_{season}_games.html"
    html = get_html(url, "#content .filter")

    if not html:
        return
    soup = BeautifulSoup(html)
    links = soup.find_all("a")
    href = [l["href"] for l in links]
    standings_pages = [f"https://www.basketball-reference.com{l}" for l in href]

    # --- NEW LOGIC: Figure out the current and previous month ---
    now = datetime.now()
    current_month = now.strftime("%B").lower() # e.g., 'march'
    
    # Safely get the previous month (handles January wrapping back to December)
    first_day_this_month = now.replace(day=1)
    prev_month = (first_day_this_month - timedelta(days=1)).strftime("%B").lower()
    
  
    for url in standings_pages:
        save_path = os.path.join(STANDINGS_DIR, url.split("/")[-1])

        # Extract the month from the URL (e.g., 'february')
        month_in_url = url.split("-")[-1].replace(".html", "")
        
        # Force a rescrape IF it's the active season AND it's the current/previous month
        force_rescrape = (season == ACTIVE_SEASON) and (month_in_url in [current_month, prev_month])

        if os.path.exists(save_path) and not force_rescrape:
            continue
        
        print(f"Scraping URL: {url}")

        html = get_html(url, "#div_schedule")
        
        if html:
            with open(save_path, "w+", encoding="utf-8") as f:
                f.write(html)
        

In [33]:
def scrape_game(standings_file):
    with open(standings_file,'r') as f:
        html = f.read()

    soup = BeautifulSoup(html)
    links = soup.find_all("a")
    hrefs = [l.get("href") for l in links]
    box_scores = [l for l in hrefs if l and "boxscores" in l and ".html" in l]
    box_scores = [f"https://www.basketball-reference.com{l}" for l in box_scores]
    for url in box_scores:
        save_path = os.path.join(SCORES_DIR,url.split("/")[-1])
        if os.path.exists(save_path):
            continue
        
        print(f"Scraping game: {url.split("/")[-1]}")

        html = get_html(url,"#content")
        if not html:
            continue

        with open(save_path,"w+",encoding="utf-8") as f:
            f.write(html)


In [34]:
def get_games():
    # if running scraper for the first time, all seasons should be scraped, not just the active one
    scrape_season(ACTIVE_SEASON)
    standings_files = os.listdir(STANDINGS_DIR)

    for f in standings_files:
        filepath = os.path.join(STANDINGS_DIR,f)
        scrape_game(filepath)

In [35]:
get_games()

Scraping URL: https://www.basketball-reference.com/leagues/NBA_2026_games-february.html
Scraping URL: https://www.basketball-reference.com/leagues/NBA_2026_games-march.html
Scraping game: 202602190CHO.html
Scraping game: 202602190CLE.html
Scraping game: 202602190PHI.html
Scraping game: 202602190WAS.html
Scraping game: 202602190NYK.html
Scraping game: 202602190CHI.html
Scraping game: 202602190SAS.html
Scraping game: 202602190GSW.html
Scraping game: 202602190SAC.html
Scraping game: 202602190LAC.html
Scraping game: 202602200CHO.html
Scraping game: 202602200MEM.html
Scraping game: 202602200WAS.html
Scraping game: 202602200ATL.html
Scraping game: 202602200MIN.html
Scraping game: 202602200NOP.html
Scraping game: 202602200OKC.html
Scraping game: 202602200LAL.html
Scraping game: 202602200POR.html
Scraping game: 202602210PHO.html
Scraping game: 202602210NOP.html
Scraping game: 202602210CHI.html
Scraping game: 202602210MIA.html
Scraping game: 202602210SAS.html
Scraping game: 202602210NYK.html
Sc